In [28]:
# Import standard libraries
from pathlib import Path
import sys
import time

# Third-party libraries
from gymnasium.utils.env_checker import check_env
import numpy as np

In [29]:
# Import the custom environment
from balance_bot_env import BalanceBotEnv

In [30]:
# Settings
MJCF_PATH = Path("../../mechanical/FreeCAD/bala2-fire/bala2-fire-simplified.xml")
SEED = 42

In [31]:
# Initialize environment
env = BalanceBotEnv(mjcf_path=MJCF_PATH, render_mode="human")

# Print observation and action spaces
print(env.observation_space)
print(env.action_space)

Box([ -3.1415927 -20.        -50.        -50.       ], [ 3.1415927 20.        50.        50.       ], (4,), float32)
Box(-1.0, 1.0, (2,), float32)


In [32]:
# Use gymnasium's built-in environment checker
check_env(env)

In [33]:
# Reset environment and print observation
obs, info = env.reset(seed=SEED)
print(obs)

[0.00155163 0.27395606 0.         0.        ]


In [34]:
# Manual step with no action (don't turn the motors)
obs, reward, terminated, truncated, info = env.step(np.array([0.0, 0.0]))
print(f"obs: {obs}")
print(f"reward: {reward}")
print(f"terminated: {terminated}")
print(f"truncated: {truncated}")

obs: [0.00308774 0.27395606 0.         0.        ]
reward: 0.9999523162834973
terminated: False
truncated: False


In [35]:
# Test: random policy
obs, _ = env.reset()
total_reward = 0

# Go for 200 steps
terminated = False
terminated = False
for step in range(200):
    step_start = time.time()

    # Choose a random action
    action = env.action_space.sample()

    # Take a step with the given action
    obs, reward, terminated, truncated, _ = env.step(action)

    # Accumulate reward
    total_reward += reward

    # Render in MuJoCo
    env.render()

    # Show that we stopped
    if terminated or truncated:
        print(f"Episode ended at step {step}")
        break

    # Make sure the loop iteration time matches the MJCF timestep time (5 ms)
    # i.e. slow the simulation down so we can see the robot behave in real time
    slack = env.model.opt.timestep - (time.time() - step_start)
    if slack > 0:
        time.sleep(slack)

print(f"Final obs: {obs}")
print(f"Final reward: {total_reward}")

Episode ended at step 110
Final obs: [ 0.55705345  5.9825616  -5.019202   -4.172037  ]
Final reward: 79.78494420672999


In [36]:
# Close the environment
env.close()